In [1]:
import os
import pandas as pd
import numpy as np
import re
import requests

In [2]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [3]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())


In [5]:
# Set browser user agent
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36 Edg/124.0.0.0'}

url = "https://www.bls.gov/cew/classifications/areas/qcew-county-msa-csa-crosswalk.xlsx"

request = requests.get(url, headers = headers)
request = request.content
df = pd.read_excel(request, sheet_name = 2, engine='openpyxl')
df

,County Code,County Title,MSA Code,MSA Title,CSA Code,CSA Title
0,1001,"Autauga County, Alabama",C3386,"Montgomery, AL MSA",NaN,NaN
1,1003,"Baldwin County, Alabama",C1930,"Daphne-Fairhope-Foley, AL MSA",CS380,"Mobile-Daphne-Fairhope, AL CSA"
2,1007,"Bibb County, Alabama",C1382,"Birmingham-Hoover, AL MSA",CS142,"Birmingham-Hoover-Talladega, AL CSA"
3,1009,"Blount County, Alabama",C1382,"Birmingham-Hoover, AL MSA",CS142,"Birmingham-Hoover-Talladega, AL CSA"
4,1015,"Calhoun County, Alabama",C1150,"Anniston-Oxford-Jacksonville, AL MSA",NaN,NaN
...,...,...,...,...,...,...
1876,72143,"Vega Alta Municipio, Puerto Rico",C4198,"San Juan-Carolina-Caguas, PR MSA",CS490,"San Juan-Carolina, PR CSA"
1877,72145,"Vega Baja Municipio, Puerto Rico",C4198,"San Juan-Carolina-Caguas, PR MSA",CS490,"San Juan-Carolina, PR CSA"
1878,72149,"Villalba Municipio, Puerto Rico",C3866,"Ponce, PR MSA",CS434,"Ponce-Coamo-Santa Isabel, PR CSA"
1879,72151,"Yabucoa Municipio, Puerto Rico",C4198,"San Juan-Carolina-Caguas, PR MSA",CS490,"San Juan-Carolina, PR CSA"


In [17]:
df[df['MSA Title'].str.contains('Fort Smith')]

,County Code,County Title,MSA Code,MSA Title,CSA Code,CSA Title
64,5033,"Crawford County, Arkansas",C2290,"Fort Smith, AR-OK MSA",NaN,NaN
88,5131,"Sebastian County, Arkansas",C2290,"Fort Smith, AR-OK MSA",NaN,NaN
1235,40079,"Le Flore County, Oklahoma",C2290,"Fort Smith, AR-OK MSA",NaN,NaN
1250,40135,"Sequoyah County, Oklahoma",C2290,"Fort Smith, AR-OK MSA",NaN,NaN


In [29]:
# Read in State Abbreviations mapping
df_states = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'StateNames')
df_states = df_states[['State', 'Postal']]

# Extract State Abbreviation from MSA label
def extract_state(text):
    return text.split(',')[1].strip()

df2 = df.copy()
df2.loc[:, 'State'] = df2['County Title'].apply(extract_state)
df2 = df2.merge(df_states, on = 'State', how = 'left')
df2[df2['Postal'].isna()]
# df2.loc[:, 'County Code'] = df2['County Code'].astype(str).apply(lambda s : s[-3:])
# df2.loc[:, 'MSA Code'   ] = df2['MSA Code'   ].apply(lambda s : s[1:] + '0')

array(['District of Columbia', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
       'Puerto Rico', 'Puerto Rico', 'Puerto Rico', 'Puerto Rico',
 

In [9]:
# Extract State Abbreviation from MSA label
def extract_state(text):
    return text.split(',')[1].strip()

df2 = df.copy()
df2.loc[:, 'State'] = df2['MSA Title'].apply(extract_state).str[:2]
df2.loc[:, 'County Code'] = df2['County Code'].astype(str).apply(lambda s : s[-3:])
df2.loc[:, 'MSA Code'   ] = df2['MSA Code'   ].apply(lambda s : s[1:] + '0')
df2 = df2.drop(['County Title', 'CSA Code', 'CSA Title'], axis = 1)
df2 = df2.rename(columns = {'County Code':'County FIPS', 'MSA Code':'MSA_ID', 'MSA Title':'MSA'})
df2 = df2.sort_values('County FIPS')
df2

,County FIPS,MSA_ID,MSA,State
0,001,33860,"Montgomery, AL MSA",AL
1283,001,23900,"Gettysburg, PA MSA",PA
93,001,41860,"San Francisco-Oakland-Hayward, CA MSA",CA
1063,001,15500,"Burlington, NC MSA",NC
1011,001,10580,"Albany-Schenectady-Troy, NY MSA",NY
...,...,...,...,...
1694,800,47260,"Virginia Beach-Norfolk-Newport News, VA-NC MSA",VA
1695,810,47260,"Virginia Beach-Norfolk-Newport News, VA-NC MSA",VA
1696,820,44420,"Staunton-Waynesboro, VA MSA",VA
1697,830,47260,"Virginia Beach-Norfolk-Newport News, VA-NC MSA",VA


In [10]:
df_fips = pd.read_excel(os.path.join(path_git, 'config', 'Area Codes.xlsx')
                            , sheet_name = 'CountyFIPS'
                            , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips = df_fips.merge(df2, on = ['State', 'County FIPS'], how = 'left')
df_fips

,State,State FIPS,County FIPS,County Name,MPO,MSA_ID,MSA
0,AL,01,001,Autauga,NaN,33860,"Montgomery, AL MSA"
1,AL,01,003,Baldwin,NaN,19300,"Daphne-Fairhope-Foley, AL MSA"
2,AL,01,005,Barbour,NaN,NaN,NaN
3,AL,01,007,Bibb,NaN,13820,"Birmingham-Hoover, AL MSA"
4,AL,01,009,Blount,NaN,13820,"Birmingham-Hoover, AL MSA"
...,...,...,...,...,...,...,...
3302,PR,72,153,Yauco Municipio,NaN,38660,"Ponce, PR MSA"
3303,UM,74,300,Midway Islands,NaN,NaN,NaN
3304,VI,78,010,St. Croix Island,NaN,NaN,NaN
3305,VI,78,020,St. John Island,NaN,NaN,NaN


In [12]:
df_fips[df_fips.duplicated(['State', 'County FIPS'])]

# df_fips[(df_fips['State'] == 'AR') & (df_fips['County FIPS'] == '079')]

,State,State FIPS,County FIPS,County Name,MPO,MSA_ID,MSA
151,AR,05,079,Lincoln,NaN,22900,"Fort Smith, AR-OK MSA"
172,AR,05,119,Pulaski,NaN,30780,"Little Rock-North Little Rock-Conway, AR MSA"
445,GA,13,113,Fayette,NaN,17980,"Columbus, GA-AL MSA"
643,IL,17,089,Kane,NaN,16980,"Chicago-Naperville-Elgin, IL-IN-WI MSA"
655,IL,17,111,McHenry,NaN,16980,"Chicago-Naperville-Elgin, IL-IN-WI MSA"
...,...,...,...,...,...,...,...
2923,VA,51,073,Gloucester,NaN,47260,"Virginia Beach-Norfolk-Newport News, VA-NC MSA"
3071,WV,54,019,Fayette,NaN,26580,"Huntington-Ashland, WV-KY-OH MSA"
3089,WV,54,053,Mason,NaN,38580,"Point Pleasant, WV-OH MicroSA"
3104,WV,54,081,Raleigh,NaN,48260,"Weirton-Steubenville, WV-OH MSA"


In [13]:
df2[df2['MSA_ID'].isin(['38220', '22900'])]

,County FIPS,MSA_ID,MSA,State
61,025,38220,"Pine Bluff, AR MSA",AR
64,033,22900,"Fort Smith, AR-OK MSA",AR
72,069,38220,"Pine Bluff, AR MSA",AR
73,079,38220,"Pine Bluff, AR MSA",AR
1235,079,22900,"Fort Smith, AR-OK MSA",AR
88,131,22900,"Fort Smith, AR-OK MSA",AR
1250,135,22900,"Fort Smith, AR-OK MSA",AR


In [15]:
df_fips[(df_fips['MSA_ID'] == '22900')]

,State,State FIPS,County FIPS,County Name,MPO,MSA_ID,MSA
127,AR,05,033,Crawford,NaN,22900,"Fort Smith, AR-OK MSA"
151,AR,05,079,Lincoln,NaN,22900,"Fort Smith, AR-OK MSA"
178,AR,05,131,Sebastian,NaN,22900,"Fort Smith, AR-OK MSA"
180,AR,05,135,Sharp,NaN,22900,"Fort Smith, AR-OK MSA"
